In [1]:
import pandas as pd
import numpy as np
import re
import time
import json
from IPython.display import JSON
import requests
from bs4 import BeautifulSoup
import unicodedata
from collections import Counter
from janome.tokenizer import Tokenizer
from janome.analyzer import Analyzer
from janome.charfilter import UnicodeNormalizeCharFilter
from janome.tokenfilter import POSKeepFilter, POSStopFilter, CompoundNounFilter, LowerCaseFilter

In [ ]:
#課題2関数実装
def get_message(file_path,start_phrase,end_phrase):
    """
    HTMLファイルから指定範囲のテキストを抽出し、形態素解析した結果を返す。
    """
    if  not "tokenizer" in globals():
        tokenizer = Tokenizer()
    char_filters=[UnicodeNormalizeCharFilter()]
    token_filters=[POSKeepFilter(['名詞']), 
                 POSStopFilter(['名詞,非自立'])]
    analyzer=Analyzer(char_filters=char_filters,tokenizer=tokenizer,token_filters=token_filters)
    with open(file_path, encoding = "utf-8") as f:
        ceremony_html = f.read()
    soup=BeautifulSoup(ceremony_html, "html.parser")
    start_index = soup.text.find(start_phrase)
    end_index = soup.text.find(end_phrase)
    if start_index == -1 or end_index == -1:
        print(f"Warning: 指定されたフレーズが {file_path} 内で見つかりませんでした。")
        return Counter()
    word_list1 = []
    word_list2 = []
    stop_words = {"兵庫", "県立", "大学", "みなさん", "たち"}
    target_text = analyzer.analyze(soup.text[start_index:end_index+len(end_phrase)])
    for i in target_text:
        word_list1.append(i.base_form)
    for j in word_list1:
        if len(j) >= 2 and j not in stop_words:
            word_list2.append(j)
    return Counter(word_list2)



In [ ]:
#課題2実行
counter_r6=get_message("./in/g-ceremony-r6.html", "感謝や友情という花言葉をもつミモザの花が春の訪れを告げる今日", "きっと大丈夫だよ。")
counter_r7=get_message("./in/ceremony-r7.html", "柔らかな春の光に包まれて、", "おめでとう。兵庫県立大学にようこそ。")
data = []
target_keywords = ["困難", "感謝", "多様", "地域", "AI"]
labels = [("令和六年度学位授与式", counter_r6), ("令和七年度入学式", counter_r7)]
for title, counter in labels:
    row = {
        "式辞": title,
        "名詞数": counter.total()
    }
    for kw in target_keywords:
        row[kw] = counter[kw]
    data.append(row)
df_message=pd.DataFrame(data)
df_message.to_csv("./out/ceremony_message.csv",index=False,encoding="utf-8")

In [1]:
#課題3関数実装
def analyze_shops(file_path,area_name):
    with open(file_path,encoding = "utf-8") as f:
        text = f.read()
    json_text = json.loads(text)
    count = 0
    sigma = 0
    word_list = []
    if  not "tokenizer" in globals():
        tokenizer = Tokenizer()
    char_filters = [UnicodeNormalizeCharFilter()]
    token_filters = [POSKeepFilter(['名詞',"一般"])]
    analyzer = Analyzer(char_filters = char_filters,tokenizer = tokenizer,token_filters = token_filters)
    for i in json_text["result"]["shop"]:
        count+=1
        match=re.search(r'～\s*(\d+)\s*円',i["budget"]["name"])
        if match:
            price=int(match.group(1))
        else:
            price=0
        sigma+=price
        for j in analyzer.analyze(i["catch"]):
            if len(j.base_form) >= 2:
                word_list.append(j.base_form)
    average=np.round(sigma/count,0)
    counter=Counter(word_list)
    counter_top=counter.most_common(4)
    result={
        "場所":area_name,
        "平均予算":average,
        "1位":counter_top[0][0] if len(counter_top) > 0 else "",
        "2位":counter_top[1][0] if len(counter_top) > 1 else "",
        "3位":counter_top[2][0] if len(counter_top) > 2 else "",
        "4位":counter_top[3][0] if len(counter_top) > 3 else ""
    }
    return result

In [2]:
#課題3実行
results=[]
results.append(analyze_shops("./in/sannomiya.json","三ノ宮"))
results.append(analyze_shops("./in/gion.json","祇園"))
df=pd.DataFrame(results)
df

FileNotFoundError: [Errno 2] No such file or directory: './in/sannomiya.json'